## 1. Importación de Librerías y Configuración del Dispositivo
En esta sección importamos las librerías necesarias (Pandas, NumPy, PyTorch y Scikit-Learn) y configuramos el dispositivo (`cuda` o `cpu`) para aprovechar la aceleración por hardware si está disponible.

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

# Configuración del dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo a utilizar: {device}")

Dispositivo a utilizar: cpu


## 2. Carga y Preprocesamiento de Datos
Cargamos el dataset ligero de COVID-19, separamos las características de la variable objetivo y realizamos una partición 80/20. Finalmente, aplicamos la normalización Z-score a nuestras características.

In [3]:
print("Cargando y Preprocesando Datos...")
archivo_covid = 'COVID_Dataset_Lab_Ligero.csv'
df = pd.read_csv(archivo_covid)

target_column = 'death_yn_Yes'
y = df[target_column].values
X_df = df.drop(['death_yn_Yes', 'death_yn_No', 'death_yn_Unknown'], axis=1)
X_raw = X_df.values

# Partición 80/20
m = len(y)
train_size = int(0.8 * m)
indices = np.arange(m)
np.random.seed(42)
np.random.shuffle(indices)

X_train = X_raw[indices][:train_size]
y_train = y[indices][:train_size]
X_test = X_raw[indices][train_size:]
y_test = y[indices][train_size:]

# Normalización Z-score
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)
sigma[sigma == 0] = 1

X_train_norm = (X_train - mu) / sigma
X_test_norm = (X_test - mu) / sigma

print(f"Forma de X_train_norm: {X_train_norm.shape}")

Cargando y Preprocesando Datos...
Forma de X_train_norm: (20000, 24)


## 3. Implementación del Dataset y DataLoader
Creamos una clase personalizada que hereda de `torch.utils.data.Dataset` para manejar nuestros datos como tensores de PyTorch. Luego instanciamos los `DataLoaders` para iterar nuestros datos en lotes (batches).

In [4]:
class DatasetPersonalizado(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        # Convertimos a tensores y los enviamos al dispositivo
        self.X = torch.from_numpy(X).float().to(device)
        self.Y = torch.from_numpy(Y).float().view(-1, 1).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

# Instanciamos los datasets
dataset_train = DatasetPersonalizado(X_train_norm, y_train)
dataset_test = DatasetPersonalizado(X_test_norm, y_test)

# Creamos el diccionario de dataloaders
dataloader = {
    'train': DataLoader(dataset_train, batch_size=64, shuffle=True),
    'test': DataLoader(dataset_test, batch_size=64, shuffle=False)
}
print("Datasets y DataLoaders creados con éxito.")

Datasets y DataLoaders creados con éxito.


## 4. Definición de la Red Neuronal
Construimos la arquitectura de nuestra red neuronal heredando de `torch.nn.Module`. Utilizamos capas lineales, una función de activación ReLU y una Sigmoide en la capa de salida para nuestra clasificación binaria.

In [5]:
class ModeloCovid(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        super(ModeloCovid, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)
        self.sigmoide = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoide(x)
        return x

## 5. Entrenamiento del Modelo y Checkpoints
Instanciamos el modelo, definimos la función de pérdida (BCELoss) y el optimizador (Adam). Ejecutamos el ciclo de entrenamiento y, al finalizar, guardamos los pesos de la red utilizando `torch.save`.

In [6]:
print("--- Entrenando el Modelo ---")
# 24 entradas, 16 neuronas ocultas, 1 salida
model = ModeloCovid(24, 16, 1).to(device)

criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 20
log_each = 5
PATH = './checkpoint_covid.pt'

model.train()
for e in range(1, epochs + 1):
    _l = []
    for x_b, y_b in dataloader['train']:
        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(_l):.5f}")

print(f"=> Guardando estado del modelo en {PATH}")
torch.save(model.state_dict(), PATH)

--- Entrenando el Modelo ---
Epoch 5/20 Loss 0.00147
Epoch 10/20 Loss 0.00146
Epoch 15/20 Loss 0.00138
Epoch 20/20 Loss 0.00135
=> Guardando estado del modelo en ./checkpoint_covid.pt


## 6. Carga del Modelo y Evaluación Final
Para comprobar que el checkpoint funciona correctamente, instanciamos un modelo completamente nuevo, le cargamos los pesos guardados y evaluamos su exactitud con el conjunto de datos de prueba (20%).

In [7]:
print("--- Validación Final con el Modelo Cargado ---")

# Instanciamos un modelo nuevo
modelo_cargado = ModeloCovid(24, 16, 1).to(device)
# Cargamos los pesos guardados
modelo_cargado.load_state_dict(torch.load(PATH))

def evaluate(model_eval, dl_test):
    model_eval.eval()
    y_true_list = []
    y_pred_list = []

    with torch.no_grad():
        for x_b, y_b in dl_test:
            y_pred = model_eval(x_b)
            predicciones_binarias = (y_pred >= 0.5).float()

            y_true_list.extend(y_b.cpu().numpy())
            y_pred_list.extend(predicciones_binarias.cpu().numpy())

    return accuracy_score(y_true_list, y_pred_list)

exactitud = evaluate(modelo_cargado, dataloader['test'])
print(f"Exactitud del modelo cargado en el set de prueba: {exactitud * 100:.2f}%")

--- Validación Final con el Modelo Cargado ---
Exactitud del modelo cargado en el set de prueba: 99.96%
